# Nested Data and Time-to-Event: Multilevel Models and Survival Analysis

Social science data often exhibits two structural patterns that lead naive regression to produce incorrect answers. The first is **nesting**: students nested within schools, employees nested within firms, respondents nested within countries. Students in the same school share principal, faculty composition, and student population—cluster-level shocks that violate independence. If you ignore this structure and naively pool 600 students as 600 independent observations into OLS, standard errors will be systematically underestimated and between-cluster variance will be misattributed to individual noise. The remedy is **hierarchical linear modeling (HLM)**—assign each cluster a random intercept (and random slopes if warranted), and quantify between-cluster variance using the **intraclass correlation coefficient (ICC)**.

The second is **right-censored time-to-event data**: we observe the time elapsed until some event—reemployment after job loss, firm failure time after founding, disease recurrence time after diagnosis. The problem is that when the observation window closes, many individuals have not yet experienced the event (right censoring); we only know "time elapsed up to now without the event occurring." Treating censoring times as event times in OLS biases estimates downward. The remedy is **event-history analysis / survival analysis**: estimate log hazard ratios (log-HR) for each covariate using the Cox proportional hazards model, and plot stratified survival curves using Kaplan-Meier. Cox relies on a critical assumption—**proportional hazards (PH)**—that hazard ratios remain constant over time. This assumption must be tested explicitly; if violated, coefficients cannot be interpreted as stable effects.

This notebook walks through both analysis pipelines. Both datasets are **synthetic data with known parameters**, allowing us to directly compare recovered estimates against ground truth—the gold standard for pedagogical notebooks. The true fixed slope in the multilevel section is 2.0 and true ICC ≈ 0.5; the true log-HR(x) = 0.8 in the survival section (i.e., HR ≈ 2.23: each unit increase in x multiplies risk by ~2.2). We use `socialverse` to execute both workflows end-to-end. It registers each method in a function registry with algorithms grounded in `statsmodels`—the reference implementations matched against R's `lme4::lmer` for multilevel analysis, and `survival::coxph` and Python's `lifelines` for survival analysis.

In [1]:
import os
import sys

# 确保用的是本 worktree 里的 socialverse(而不是环境里 editable 安装指向的其它 checkout)
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # 在 Jupyter cell 里没有 __file__,退回当前工作目录
    _HERE = os.path.abspath(os.getcwd())
_ROOT = os.path.dirname(_HERE) if os.path.basename(_HERE) == "notebooks" else _HERE
if os.path.isdir(os.path.join(_ROOT, "socialverse")) and _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件
import matplotlib.pyplot as plt
from matplotlib import font_manager as _fm
import numpy as np
import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

# 让本 notebook 自绘的图也能显示中文标签
_CJK = ["PingFang SC", "Hiragino Sans GB", "Songti SC", "STHeiti",
        "Arial Unicode MS", "Noto Sans CJK SC", "Microsoft YaHei"]
_have = {f.name for f in _fm.fontManager.ttflist}
plt.rcParams["font.sans-serif"] = [c for c in _CJK if c in _have] + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False  # 用 ASCII 连字符,负号正常显示

# Part 1 · Multilevel Models: Students Nested Within Schools

## Loading Nested Data

We use a built-in two-level synthetic dataset: 30 schools × 20 students per school = 600 observations. The data-generating process follows a random-intercept model: `y = 1.0 + u_school + 2.0·x + noise`, where `u_school ~ N(0, 1²)` is the school-level random intercept (cluster shock) and `x` is a student-level predictor. The true fixed slope is 2.0; cluster and residual variances are both approximately 1, yielding true ICC ≈ 0.5—exactly half the outcome variance lies between schools. This is the canonical scenario where ignoring nesting produces biased inference.

Data are in long format, one row per student: `school` identifies the cluster, `student` the within-cluster ID, `x` is the predictor, and `y` is the outcome.

In [2]:
mldf = ds.load_multilevel()
print("形状:", mldf.shape, "| 学校数:", mldf["school"].nunique(),
      "| 每校学生数:", mldf.groupby("school").size().iloc[0])
mldf.head()

形状: (600, 4) | 学校数: 30 | 每校学生数: 20


,school,student,x,y
0,0,0,-0.1321,1.5019
1,0,1,0.1049,0.7999
2,0,2,0.3616,3.1529
3,0,3,0.9471,2.3162
4,0,4,-1.2654,-2.0284


## Ingesting into Study State

As with all analyses, the first step is not fitting models but specifying the data and outcome variable. `sv.pp.ingest` registers the table in a study state, and `variables.outcome` declares the outcome column as `y`. Downstream calls to `multilevel` will retrieve the outcome name from this contract—no need to pass it again.

In [3]:
st = sv.StudyState()
sv.pp.ingest(st, data=mldf, name="students_in_schools")
st.write("variables", "outcome", "y")   # 结果变量是 y
st.populated()

{'sources': ['datasets', 'dataset_name', 'dataset_meta'],
 'variables': ['outcome']}

## Fitting the Random-Intercept Model

`multilevel` fits `y ~ 1 + x` with school-level random intercepts using `statsmodels.MixedLM` (REML). Specify clustering via `groups` and fixed-effect predictors via `predictors`. The function returns two outputs: fixed-effect coefficients (with standard errors, cluster count, and convergence status), and variance components—cluster variance, residual variance, and the computed ICC.

In [4]:
sv.tl.multilevel(st, groups="school", predictors=["x"])

mm = st.models["mixedlm"]
vc = st.diagnostics["variance_components"]

print("估计器:", mm["estimator"], "| 收敛:", mm["converged"],
      "| n:", mm["n"], "| 组数:", mm["n_groups"])
print("\n固定效应 (系数, SE):")
for name, (b, se) in mm["fixed_effects"].items():
    print(f"  {name:<10} {b:+.4f}  (SE {se:.4f})")
print("\n方差成分:")
print(f"  组间方差 σ²_u = {vc['between_var']:.4f}")
print(f"  残差方差 σ²_e = {vc['residual_var']:.4f}")
print(f"  ICC          = {vc['icc']:.4f}")

估计器: statsmodels.MixedLM(REML) | 收敛: True | n: 600 | 组数: 30

固定效应 (系数, SE):
  Intercept  +1.2074  (SE 0.1954)
  x          +1.9896  (SE 0.0417)

方差成分:
  组间方差 σ²_u = 1.0384
  残差方差 σ²_e = 0.9458
  ICC          = 0.5233


**Interpretation:** the fixed slope for `x` is estimated at **+1.99**, nearly exactly the true value of 2.0; the intercept is approximately 1.2. Variance components show cluster and residual variances (≈1.04 and ≈0.95) are roughly equal, yielding ICC ≈ **0.52**—about half of outcome variance lies between schools, exactly as designed.

Since this is synthetic data with known parameters, we can directly compare recovered estimates to ground truth. Tighter agreement indicates more accurate estimation.

In [5]:
slope_hat = mm["fixed_effects"]["x"][0]
icc_hat = vc["icc"]
pd.DataFrame([
    {"量": "固定斜率 x", "真值": 2.0, "估计": round(slope_hat, 3), "绝对误差": round(abs(slope_hat - 2.0), 3)},
    {"量": "ICC",       "真值": 0.5, "估计": round(icc_hat, 3),   "绝对误差": round(abs(icc_hat - 0.5), 3)},
])

,量,真值,估计,绝对误差
0,固定斜率 x,2.0,1.990,0.010
1,ICC,0.5,0.523,0.023


Slope and ICC both cluster around their true values—the between-cluster variance under the multilevel structure has been correctly separated.

## Visualization: Geometric Intuition for Random Intercepts

What does ICC ≈ 0.5 look like? Below, we fit `y ~ x` separately for each school (thin colored lines) and overlay the HLM-estimated fixed-effect line (thick red line). If the data truly follow a random-intercept structure, school-specific lines should exhibit **similar slopes but shifted intercepts**—shared slopes (all near 2) with intercepts that drift, manifesting the random intercept `u_school` and illustrating HLM's advantage over naive pooled OLS.

In [6]:
fig, ax = plt.subplots(figsize=(7.0, 4.6))
cmap = plt.get_cmap("viridis")
schools = sorted(mldf["school"].unique())
for i, g in enumerate(schools):
    sub = mldf[mldf["school"] == g]
    b1, b0 = np.polyfit(sub["x"].to_numpy(), sub["y"].to_numpy(), 1)  # 每校自己的 OLS 线
    xs = np.linspace(sub["x"].min(), sub["x"].max(), 20)
    ax.plot(xs, b0 + b1 * xs, color=cmap(i / max(len(schools) - 1, 1)),
            alpha=0.55, linewidth=1.0, zorder=2)
# 叠加 HLM 固定效应线:固定截距 + 固定斜率
b0_fix = mm["fixed_effects"]["Intercept"][0]
b1_fix = mm["fixed_effects"]["x"][0]
xs = np.linspace(mldf["x"].min(), mldf["x"].max(), 50)
ax.plot(xs, b0_fix + b1_fix * xs, color="crimson", linewidth=3.0, zorder=5,
        label=f"HLM 固定效应线 (斜率 {b1_fix:.2f})")
ax.set_xlabel("学生级预测变量 x")
ax.set_ylabel("结果 y")
ax.set_title(f"30 所学校各自的回归线:随机截距上下平移(ICC ≈ {icc_hat:.2f})")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout()
fig.savefig("fig_hlm_schools.png", dpi=150)
plt.close(fig)
print("已存 fig_hlm_schools.png")

已存 fig_hlm_schools.png


![School Random Intercepts](fig_hlm_schools.png)

Thin lines show school-specific fitted lines (colored by school ID); the thick red line is the HLM fixed-effect line. School-specific lines are roughly **parallel** (slopes near 2) but **vertically shifted**—this is the visual signature of random intercepts `u_school`.

# Part 2 · Survival Analysis: Time-to-Event and Right Censoring

## Loading Survival Data

`ds.load_survival(beta=0.8)` generates 400 records from an exponential proportional-hazards model with hazard rate `λ(t) = 0.1·exp(0.8·x + 0.5·group)`. Observed time is the minimum of event and censoring times; `event = 1` only when the event occurred before censoring. The true **log-HR(x) = 0.8** (i.e., HR ≈ 2.23: each unit increase in x multiplies risk by ~2.2).

Each row is one individual: `time` is observed follow-up duration, `event` is the event indicator (1 = observed, 0 = censored), and `x` and `group` are covariates. Note that a substantial fraction is censored—this is precisely why OLS fails and survival models are essential.

In [7]:
survdf = ds.load_survival(beta=0.8)
n_evt = int(survdf["event"].sum())
print("形状:", survdf.shape,
      f"| 事件数 = {n_evt} / {len(survdf)}  "
      f"(删失 {len(survdf) - n_evt} 条,删失率 {1 - n_evt / len(survdf):.0%})")
survdf.head()

形状: (400, 4) | 事件数 = 290 / 400  (删失 110 条,删失率 28%)


,time,event,x,group
0,0.0580,1,0.1257,0
1,5.3783,1,-0.1321,0
2,1.7805,1,0.6404,1
3,1.3061,1,0.1049,1
4,1.5218,1,-0.5357,1


## Fitting the Cox Model

Ingest the data into a new study state, specifying `time` as the follow-up duration column. Then call `survival`, with arguments: `time` for duration, `event` for the event indicator, and `covariates` for predictors. The function returns three outputs: Cox log-HR estimates (via `statsmodels.PHReg` with Breslow ties handling), Kaplan-Meier survival curves, and a proportional-hazards assumption test. We first examine the Cox coefficients.

In [8]:
st2 = sv.StudyState()
sv.pp.ingest(st2, data=survdf, name="time_to_event")
st2.write("variables", "outcome", "time")   # 时长列
sv.tl.survival(st2, time="time", event="event", covariates=["x", "group"])

cox = st2.models["cox"]
print("估计器:", cox["estimator"], "| n:", cox["n"], "| 事件数:", cox["n_events"])
print("\n协变量        log-HR      SE       HR        p")
for name, (b, se, p) in cox["log_hr"].items():
    print(f"  {name:<10} {b:+.4f}   {se:.4f}   {cox['hr'][name]:.3f}   {p:.2e}")

估计器: statsmodels.PHReg (Cox, Breslow ties) | n: 400 | 事件数: 290

协变量        log-HR      SE       HR        p
  x          +0.8340   0.0702   2.303   1.56e-32
  group      +0.5829   0.1207   1.791   1.37e-06


**Interpretation:** the log-HR for `x` is estimated at ≈ **0.83** (true value 0.8), yielding HR ≈ **2.30**—each unit increase in x multiplies the hazard by ~2.3. The HR for `group` is ≈ 1.79, also significant. Both p-values are far below 0.05. Compare the `x` estimate to its true value:

In [9]:
loghr_x = cox["log_hr"]["x"][0]
pd.DataFrame([
    {"量": "log-HR(x)",     "真值": round(0.8, 3),         "估计": round(loghr_x, 3),        "绝对误差": round(abs(loghr_x - 0.8), 3)},
    {"量": "HR(x)=exp(·)",  "真值": round(float(np.exp(0.8)), 3), "估计": round(cox["hr"]["x"], 3), "绝对误差": round(abs(cox["hr"]["x"] - np.exp(0.8)), 3)},
])

,量,真值,估计,绝对误差
0,log-HR(x),0.800,0.834,0.034
1,HR(x)=exp(·),2.226,2.303,0.077


Despite ~28% censoring, Cox partial likelihood recovers the true log-HR—this is its key advantage over OLS: censored observations are neither discarded nor treated as events, but rather contribute as "at-risk" until their censoring time.

## Testing the Proportional-Hazards Assumption

Whether Cox coefficients can be interpreted as stable hazard ratios depends on the **proportional-hazards (PH) assumption**: hazard ratios must remain constant over time. This is Cox's "parallel trends" condition—violation means coefficients cannot be interpreted as time-stable effects. The `survival` function automatically runs the Grambsch-Therneau test (equivalent to R's `survival::cox.zph`): it tests whether Schoenfeld residuals correlate with event times for each covariate, then aggregates into a global test. `p > 0.05` indicates non-rejection of PH—the assumption holds.

In [10]:
ph = st2.diagnostics["ph_test"]
print("方法:", ph["method"])
print(f"全局 PH 检验: χ² = {ph['global_chi2']:.3f}, p = {ph['global_p']:.3f} → 判定: {ph['verdict']}")
print("\n逐协变量:")
for name, d in ph["per_covariate"].items():
    print(f"  {name:<8} rho={d['rho']:+.4f}  χ²={d['chi2']:.3f}  p={d['p']:.3f}")

方法: Grambsch-Therneau on scaled Schoenfeld residuals (cox.zph, transform=rank)
全局 PH 检验: χ² = 0.866, p = 0.648 → 判定: pass

逐协变量:
  x        rho=+0.0472  χ²=0.643  p=0.423
  group    rho=-0.0278  χ²=0.223  p=0.637


The global test yields `p ≈ 0.65`, well above 0.05—no rejection of proportional hazards; no individual covariate shows violation. The assumption holds, and Cox log-HR estimates can be interpreted as stable hazard ratios.

## Visualization: Kaplan-Meier Survival Curves

First, examine median survival times estimated by KM: overall and stratified by `group`. Because `group=1` has an additional `+0.5` term in the hazard (higher risk), its survival curve should decline more rapidly and exhibit shorter median survival.

In [11]:
km = st2.models["km"]
print("KM 分组列:", km["group_col"], "| 分组:", sorted(km["by_group"].keys()))
print(f"总体中位生存时间: {km['overall']['median']:.3f}")
for g, curve in sorted(km["by_group"].items()):
    print(f"  组 {g}: 中位生存时间 = {curve['median']:.3f}  (n={curve['n']})")

KM 分组列: group | 分组: ['0', '1']
总体中位生存时间: 4.954
  组 0: 中位生存时间 = 6.678  (n=193)
  组 1: 中位生存时间 = 3.736  (n=207)


`sv.pl.km_curve` reads KM curves from the study state and plots them as right-continuous step functions (standard Kaplan-Meier visualization), comparable to `survminer::ggsurvplot` and `lifelines.KaplanMeierFitter.plot`.

In [12]:
sv.pl.km_curve(st2, out="fig_km.png", title="Kaplan-Meier 生存曲线(按 group 分层)")
print("图已登记:", st2.artifacts["figures"]["km"])

图已登记: {'path': 'fig_km.png', 'dpi': 200, 'note': '2 条 KM 生存阶梯曲线'}


![KM Survival Curves](fig_km.png)

Solid lines show stratified survival curves for `group=0/1`; the gray dashed line is the overall reference. The `group=1` curve (high-risk) declines more steeply, with median survival (≈3.7) notably shorter than `group=0` (≈6.7)—consistent with the data-generating mechanism.

## Visualization: Hazard Ratio Forest Plot

Finally, plot hazard ratios for both covariates as a forest plot on the log scale, showing HR and 95% confidence intervals. The `HR=1` dashed line marks the "null" reference: when a confidence interval lies entirely to the right of 1, the covariate significantly elevates hazard.

In [13]:
names = list(cox["log_hr"].keys())
b = np.array([cox["log_hr"][k][0] for k in names])
se = np.array([cox["log_hr"][k][1] for k in names])
hr = np.exp(b)
lo = np.exp(b - 1.96 * se)   # 95% CI 下界
hi = np.exp(b + 1.96 * se)   # 95% CI 上界

fig, ax = plt.subplots(figsize=(6.6, 2.6))
ypos = np.arange(len(names))[::-1]
ax.errorbar(hr, ypos, xerr=[hr - lo, hi - hr], fmt="o", color="#2c3e70",
            capsize=4, markersize=7, linewidth=1.6, zorder=3)
ax.axvline(1.0, color="0.5", linestyle="--", linewidth=1.0, zorder=1)  # HR=1 无效应参考
ax.set_yticks(ypos)
ax.set_yticklabels(list(names))
ax.set_xscale("log")   # 风险比在对数刻度上对称
ax.set_xlabel("风险比 HR = exp(log-HR)  (对数刻度)")
ax.set_title("Cox 比例风险:HR 与 95% CI")
for x_, y_ in zip(hr, ypos):
    ax.annotate(f"{x_:.2f}", (x_, y_), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=9)
ax.set_ylim(-0.6, len(names) - 0.4)
fig.tight_layout()
fig.savefig("fig_cox_forest.png", dpi=150)
plt.close(fig)
print("已存 fig_cox_forest.png")

已存 fig_cox_forest.png


![Cox Forest Plot](fig_cox_forest.png)

The HR for `x` is ≈ 2.3 (true value exp(0.8) = 2.23); for `group` it is ≈ 1.8. Both confidence intervals lie entirely to the right of the `HR=1` dashed line → both significantly elevate hazard.

## Reproducible Evidence Chain

Both analysis pipelines run on a `StudyState`, which automatically accumulates a provenance ledger: which function was called at each step, what slots it consumed, and what it produced. This ledger is not hand-written documentation but an auto-generated record appended after each successful execution—in social science, transparency about which step produced which conclusion from which data is as critical as the conclusion itself.

In [14]:
print("=== 多层链 ===")
print(st.summary())
print("\n=== 生存链 ===")
print(st2.summary())

=== 多层链 ===
StudyState
  sources: datasets, dataset_name, dataset_meta
  variables: outcome
  models: mixedlm
  diagnostics: variance_components
  provenance: 2 step(s)

=== 生存链 ===
StudyState
  sources: datasets, dataset_name, dataset_meta
  variables: outcome
  models: cox, km
  diagnostics: ph_test
  artifacts: figures
  provenance: 3 step(s)


## Summary

We have walked through two analysis pipelines for data with special structures. The multilevel pipeline benchmarks R's `lme4::lmer` and Python's `statsmodels.MixedLM`: random intercepts absorb cluster-level shocks; ICC quantifies between-cluster variance. The survival pipeline benchmarks `survival::coxph` plus `survminer` and Python's `lifelines`: Cox partial likelihood estimates hazard ratios under censoring; Kaplan-Meier plots stratified survival curves; the proportional-hazards test guards the key assumption.

Relative to calling `statsmodels` directly, `socialverse` adds two capabilities: first, it encodes as machine-readable contracts which column is the outcome and what prerequisites each step requires, making analyses auditable by schema rather than guesswork. Second, it maintains an end-to-end, traceable evidence chain. The next tutorial, [14_spatial_analysis](14_spatial_analysis.ipynb), turns to spatial data—how to model observations when geography introduces dependence.